# Notebook 16: Defense v2 — Interpretability-Guided Improvement

**Purpose:** Build improved defenses based on the failure analysis from notebook 15. This is the 'fix' phase of the build-break-fix cycle.

**Defense improvements:**
1. **Data augmentation** — Add successful evasions to training set
2. **Ensemble defense** — SAE + TF-IDF combined (attack must evade both)
3. **Feature steering + re-scan** — Steer borderline cases, then re-classify
4. **Adaptive thresholds** — Lower thresholds for high-risk categories

**Key metric:** v1 vs v2 evasion rates across all attack strategies.

**Prerequisites:** Run notebooks 14-15 first.

**Runtime:** Colab GPU (T4), ~25 minutes.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -r /content/drive/MyDrive/iris/requirements.txt -q

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/iris')

import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

from src.utils.helpers import set_seed, get_device
set_seed(42)
device = get_device()

DRIVE_ROOT = Path('/content/drive/MyDrive/iris')

Using GPU: NVIDIA L4


## Step 0: Load Everything + Reproduce v1 Baseline

In [ ]:
from src.model.transformer import load_model, extract_activations
from src.sae.architecture import SparseAutoencoder
from src.utils.helpers import load_checkpoint
from src.data.dataset import IrisDataset, SYSTEM_PROMPT_TEMPLATE
from src.data.preprocessing import tokenize_prompts
from src.analysis.features import compute_feature_activations
from src.analysis.red_team import generate_red_team_suite, evaluate_red_team
from src.analysis.adversarial import generate_evasion_prompts
from src.baseline.classifiers import train_tfidf_baseline
from sklearn.linear_model import LogisticRegression

# Load models
gpt2 = load_model(device)
sae = SparseAutoencoder(d_input=1280, expansion_factor=8, sparsity_coeff=1e-4)
load_checkpoint(DRIVE_ROOT / 'checkpoints' / 'sae_d10240_lambda1e-04.pt', sae, device=device)
sae = sae.to(device).eval()

# Read target layer from J2 metrics
with open(str(DRIVE_ROOT / 'results' / 'metrics' / 'j2_evaluation.json')) as f:
    j2_metrics = json.load(f)
TARGET_LAYER = j2_metrics['train_layer']
print(f'Target layer: {TARGET_LAYER}')

# Load dataset — MUST use balanced dataset because feature_matrix.npy was
# computed from it. Using expanded would cause a label/feature mismatch.
ds_path = DRIVE_ROOT / 'data' / 'processed' / 'iris_dataset_balanced.json'
dataset = IrisDataset.load(ds_path)
labels = np.array(dataset.labels)

feature_matrix = np.load(str(DRIVE_ROOT / 'checkpoints' / 'feature_matrix.npy'))
sensitivity = np.load(str(DRIVE_ROOT / 'checkpoints' / 'sensitivity_scores.npy'))

assert feature_matrix.shape[0] == len(labels), (
    f'Mismatch: feature_matrix has {feature_matrix.shape[0]} rows but dataset has {len(labels)} examples'
)

# --- Two attack sets ---
# Training attacks (seed=42): used to find evasions and augment v2 training set
# Held-out test attacks (seed=123): used to evaluate BOTH v1 and v2 fairly
# This avoids data leakage: v2 is never tested on attacks it trained on.
train_rt = generate_red_team_suite(n_per_strategy=20, seed=42)
train_evasions = generate_evasion_prompts(n=50, seed=42)
train_attacks = train_rt + train_evasions

test_rt = generate_red_team_suite(n_per_strategy=20, seed=123)
test_evasions = generate_evasion_prompts(n=50, seed=123)
test_attacks = test_rt + test_evasions

print(f'Dataset: {len(dataset)} examples')
print(f'Training attacks: {len(train_attacks)} (seed=42, for augmentation)')
print(f'Held-out test attacks: {len(test_attacks)} (seed=123, for evaluation)')

In [ ]:
def extract_features_batch(texts, batch_size=16):
    """Extract SAE features for a list of texts."""
    formatted = [SYSTEM_PROMPT_TEMPLATE.format(prompt=t) for t in texts]
    all_features = []
    for i in range(0, len(formatted), batch_size):
        batch = formatted[i:i+batch_size]
        tokenized = tokenize_prompts(batch, max_length=128)
        acts = extract_activations(gpt2, tokenized['input_ids'], tokenized['attention_mask'],
                                   layers=[TARGET_LAYER], batch_size=batch_size)
        feats = compute_feature_activations(sae, acts[TARGET_LAYER], device=device)
        all_features.append(feats)
    return np.vstack(all_features)

# v1 detector (baseline)
v1_detector = LogisticRegression(max_iter=1000, random_state=42)
v1_detector.fit(feature_matrix, labels)

# Extract features for BOTH attack sets
print('Extracting features for training attacks...')
train_texts = [a['text'] for a in train_attacks]
train_features = extract_features_batch(train_texts)

print('Extracting features for held-out test attacks...')
test_texts = [a['text'] for a in test_attacks]
test_features = extract_features_batch(test_texts)

# v1 predictions on training set (to identify evasions for augmentation)
v1_train_preds = v1_detector.predict(train_features)
v1_train_evaded = sum(1 for p in v1_train_preds if p == 0)
print(f'v1 on training attacks: {v1_train_evaded}/{len(v1_train_preds)} evaded ({v1_train_evaded/len(v1_train_preds):.1%})')

# v1 predictions on held-out test set (baseline for fair comparison)
v1_preds = v1_detector.predict(test_features)
v1_probs = v1_detector.predict_proba(test_features)[:, 1]
v1_evaded = sum(1 for p in v1_preds if p == 0)
print(f'v1 on held-out test:    {v1_evaded}/{len(v1_preds)} evaded ({v1_evaded/len(v1_preds):.1%})')

## Defense v2a: Data Augmentation

Add successful evasions to the training set and retrain the detector.

In [ ]:
# Identify v1 evasions from TRAINING set (seed=42)
evaded_mask = v1_train_preds == 0
evaded_features = train_features[evaded_mask]

print(f'Adding {sum(evaded_mask)} successful evasions from training set to detector...')

# Augment training data with evasions
augmented_features = np.vstack([feature_matrix, evaded_features])
augmented_labels = np.concatenate([labels, np.ones(sum(evaded_mask), dtype=int)])

print(f'Augmented dataset: {len(augmented_labels)} examples '
      f'({sum(augmented_labels)} injection, {len(augmented_labels) - sum(augmented_labels)} normal)')

# Train v2a detector
v2a_detector = LogisticRegression(max_iter=1000, random_state=42)
v2a_detector.fit(augmented_features, augmented_labels)

# Evaluate v2a on HELD-OUT test set (seed=123) — no data leakage
v2a_preds = v2a_detector.predict(test_features)
v2a_evaded = sum(1 for p in v2a_preds if p == 0)
print(f'v2a on held-out test: {v2a_evaded}/{len(v2a_preds)} evaded ({v2a_evaded/len(v2a_preds):.1%})')
print(f'Improvement over v1: {v1_evaded - v2a_evaded} fewer evasions')

# Check false positive rate on normal training data
normal_feats = feature_matrix[labels == 0]
v1_fp = sum(v1_detector.predict(normal_feats))
v2a_fp = sum(v2a_detector.predict(normal_feats))
print(f'False positives on normal data: v1={v1_fp}, v2a={v2a_fp}')

## Defense v2b: Ensemble Detection (SAE + TF-IDF)

Combine SAE and TF-IDF detection. An input is flagged only if it evades *both* — defense-in-depth.

In [ ]:
# Train TF-IDF detector on original dataset
lr_pipeline, _ = train_tfidf_baseline(dataset.texts, dataset.labels, seed=42)

# TF-IDF predictions on held-out test set
tfidf_preds = lr_pipeline.predict(test_texts)
tfidf_probs = lr_pipeline.predict_proba(test_texts)[:, 1]

# Ensemble: OR logic (flagged if EITHER detector catches it)
ensemble_preds = np.maximum(v1_preds, tfidf_preds)
ensemble_evaded = sum(1 for p in ensemble_preds if p == 0)

print('Ensemble (SAE OR TF-IDF) on held-out test set:')
print(f'  SAE alone:      {v1_evaded}/{len(v1_preds)} evaded ({v1_evaded/len(v1_preds):.1%})')
print(f'  TF-IDF alone:   {sum(1 for p in tfidf_preds if p == 0)}/{len(tfidf_preds)} evaded')
print(f'  Ensemble (OR):  {ensemble_evaded}/{len(ensemble_preds)} evaded ({ensemble_evaded/len(ensemble_preds):.1%})')
print()

# Per-strategy comparison
print('Per-strategy evasion rates (held-out test):')
print(f'{"Strategy":>25s} {"SAE":>8s} {"TF-IDF":>8s} {"Ensemble":>10s}')
print('-' * 55)

strategies = sorted(set(a.get('evasion_strategy', 'unknown') for a in test_attacks))
for strategy in strategies:
    indices = [i for i, a in enumerate(test_attacks) if a.get('evasion_strategy') == strategy]
    sae_evade = sum(1 for i in indices if v1_preds[i] == 0)
    tfidf_evade = sum(1 for i in indices if tfidf_preds[i] == 0)
    ens_evade = sum(1 for i in indices if ensemble_preds[i] == 0)
    n = len(indices)
    print(f'{strategy:>25s} {sae_evade/n:>7.0%} {tfidf_evade/n:>7.0%} {ens_evade/n:>9.0%}')

## Defense v2c: Combined (Augmentation + Ensemble)

The strongest defense: augmented SAE detector + TF-IDF ensemble.

In [ ]:
# Combined: augmented SAE OR TF-IDF (on held-out test set)
v2c_preds = np.maximum(v2a_preds, tfidf_preds)
v2c_evaded = sum(1 for p in v2c_preds if p == 0)

print(f'Combined defense (augmented SAE + TF-IDF) on held-out test:')
print(f'  Evasions: {v2c_evaded}/{len(v2c_preds)} ({v2c_evaded/len(v2c_preds):.1%})')
print(f'  vs v1: {v1_evaded} -> {v2c_evaded} ({v1_evaded - v2c_evaded} fewer)')

## Defense v2d: Feature Steering + Re-scan

For borderline cases (0.3 < probability < 0.7), apply steering to suppress injection features, then re-classify.

In [ ]:
from src.agent.steering import SteeringDefense

steering = SteeringDefense(
    sae_model=sae,
    sensitivity_scores=sensitivity,
    gpt2_model=gpt2,
    detector=v2a_detector,  # Use augmented detector
    top_k=20,
    layer=TARGET_LAYER,
)

# Identify borderline cases from v2a on held-out test set
v2a_probs = v2a_detector.predict_proba(test_features)[:, 1]
borderline_mask = (v2a_probs > 0.3) & (v2a_probs < 0.7)
borderline_indices = np.where(borderline_mask)[0]

print(f'Borderline cases (0.3 < prob < 0.7): {len(borderline_indices)}')

# Apply steering to borderline cases
steering_upgrades = 0
for idx in borderline_indices[:20]:  # Limit for runtime
    text = test_texts[idx]
    result = steering.dampen(text, scale=0.0)
    
    # If steering pushes it above threshold, upgrade to detection
    if result['steered_prob'] < 0.3:  # Steering neutralized it -> it WAS benign
        pass
    elif result['orig_prob'] < 0.5 and result['steered_prob'] < result['orig_prob']:
        # Was below threshold, steering reduced it further -> likely benign
        pass
    else:
        steering_upgrades += 1

print(f'Steering-based upgrades: {steering_upgrades}/{len(borderline_indices[:20])}')

## v1 vs v2 Comparison

In [ ]:
from src.analysis.evaluation import compute_detection_metrics

# All evaluations on held-out test set (seed=123) for fair comparison
y_true = np.ones(len(test_attacks), dtype=int)  # All attacks are label=1

v1_metrics = compute_detection_metrics(y_true, v1_preds, v1_probs)
v2a_metrics = compute_detection_metrics(y_true, v2a_preds, v2a_probs)

tfidf_only_metrics = compute_detection_metrics(y_true, tfidf_preds, tfidf_probs)
ensemble_metrics = compute_detection_metrics(y_true, ensemble_preds)
v2c_metrics = compute_detection_metrics(y_true, v2c_preds)

print('=' * 70)
print('  v1 vs v2 Defense Comparison (held-out test set)')
print('=' * 70)
print(f'{"Defense":<30s} {"Det Rate":>10s} {"Evasion":>10s} {"Precision":>10s}')
print('-' * 70)
print(f'{"v1 (SAE baseline)":<30s} {v1_metrics["detection_rate"]:>9.1%} {1-v1_metrics["detection_rate"]:>9.1%} {v1_metrics["precision"]:>9.1%}')
print(f'{"v2a (SAE + augmentation)":<30s} {v2a_metrics["detection_rate"]:>9.1%} {1-v2a_metrics["detection_rate"]:>9.1%} {v2a_metrics["precision"]:>9.1%}')
print(f'{"TF-IDF alone":<30s} {tfidf_only_metrics["detection_rate"]:>9.1%} {1-tfidf_only_metrics["detection_rate"]:>9.1%} {tfidf_only_metrics["precision"]:>9.1%}')
print(f'{"Ensemble (SAE OR TF-IDF)":<30s} {ensemble_metrics["detection_rate"]:>9.1%} {1-ensemble_metrics["detection_rate"]:>9.1%} {ensemble_metrics["precision"]:>9.1%}')
print(f'{"v2c (augmented + ensemble)":<30s} {v2c_metrics["detection_rate"]:>9.1%} {1-v2c_metrics["detection_rate"]:>9.1%} {v2c_metrics["precision"]:>9.1%}')
print('=' * 70)
print()
print('Note: All results on held-out test attacks (seed=123), not the')
print('training attacks used for augmentation (seed=42). No data leakage.')

## Visualization: v1 vs v2 Per-Strategy

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

strategies = sorted(set(a.get('evasion_strategy', 'unknown') for a in test_attacks))
x = np.arange(len(strategies))
width = 0.3

v1_rates = []
v2c_rates = []
for strategy in strategies:
    indices = [i for i, a in enumerate(test_attacks) if a.get('evasion_strategy') == strategy]
    n = len(indices)
    v1_rates.append(sum(1 for i in indices if v1_preds[i] == 0) / n)
    v2c_rates.append(sum(1 for i in indices if v2c_preds[i] == 0) / n)

bars1 = ax.bar(x - width/2, v1_rates, width, label='v1 (SAE baseline)', color='#D55E00', alpha=0.85)
bars2 = ax.bar(x + width/2, v2c_rates, width, label='v2 (augmented + ensemble)', color='#0072B2', alpha=0.85)

ax.set_xlabel('Attack Strategy', fontsize=12)
ax.set_ylabel('Evasion Rate', fontsize=12)
ax.set_title('Defense v1 vs v2: Per-Strategy Evasion Rates (held-out test)', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(strategies, rotation=45, ha='right', fontsize=9)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.15)
ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=1, alpha=0.3)

# Add value labels
for bar in bars1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.02, f'{h:.0%}', ha='center', fontsize=8)
for bar in bars2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.02, f'{h:.0%}', ha='center', fontsize=8)

plt.tight_layout()
save_path = DRIVE_ROOT / 'results' / 'figures' / 'defense_v1_vs_v2.png'
fig.savefig(str(save_path), dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved to {save_path}')

## Save v2 Results

In [ ]:
v2_results = {
    'experiment': 'defense_v2',
    'evaluation_set': 'held-out test (seed=123)',
    'n_test_attacks': len(test_attacks),
    'n_train_attacks': len(train_attacks),
    'v1_evasion_rate': v1_evaded / len(v1_preds),
    'v2a_augmented_evasion_rate': v2a_evaded / len(v2a_preds),
    'ensemble_evasion_rate': ensemble_evaded / len(ensemble_preds),
    'v2c_combined_evasion_rate': v2c_evaded / len(v2c_preds),
    'v1_metrics': {k: float(v) if isinstance(v, (int, float, np.floating, np.integer)) else v
                   for k, v in v1_metrics.items()},
    'v2c_metrics': {k: float(v) if isinstance(v, (int, float, np.floating, np.integer)) else v
                    for k, v in v2c_metrics.items()},
    'per_strategy_v1': {s: sum(1 for i in [j for j, a in enumerate(test_attacks) if a.get('evasion_strategy') == s] if v1_preds[i] == 0) / max(sum(1 for a in test_attacks if a.get('evasion_strategy') == s), 1) for s in strategies},
    'per_strategy_v2c': {s: sum(1 for i in [j for j, a in enumerate(test_attacks) if a.get('evasion_strategy') == s] if v2c_preds[i] == 0) / max(sum(1 for a in test_attacks if a.get('evasion_strategy') == s), 1) for s in strategies},
    'augmentation_size': int(sum(evaded_mask)),
    'n_borderline': int(len(borderline_indices)),
}

results_path = DRIVE_ROOT / 'results' / 'metrics' / 'defense_v2.json'
results_path.parent.mkdir(parents=True, exist_ok=True)
results_path.write_text(json.dumps(v2_results, indent=2))
print(f'Results saved to {results_path}')

print()
print('=' * 60)
print('  Defense Engineering Cycle — Complete')
print('=' * 60)
print()
print('Build → Break → Fix:')
print(f'  v1 evasion rate: {v1_evaded/len(v1_preds):.1%}')
print(f'  v2 evasion rate: {v2c_evaded/len(v2c_preds):.1%}')
improvement = (v1_evaded - v2c_evaded) / max(v1_evaded, 1)
print(f'  Improvement:     {improvement:.0%} reduction in evasions')
print()
print('Note: All rates measured on held-out test attacks (seed=123).')
print('Remaining gaps should be documented honestly — intellectual')
print('maturity matters more than perfect scores.')